In [2]:
import gensim.downloader as api

In [3]:
model_w2v = api.load("word2vec-google-news-300")

[==================================================] 100.0% 1662.8/1662.8MB downloaded


In [4]:
model_glove = api.load("glove-wiki-gigaword-300")

[==================================================] 100.0% 376.1/376.1MB downloaded


In [25]:
def load_analogies(file_path):
  """Get the groups from the provided datast file"""
  groups = {}
  current_group = None

  with open(file_path, 'r', encoding='utf-8') as f:
      for line in f:
          line = line.strip()

          # Skip empty lines or copyright/header lines
          if not line or line.startswith("//"):
              continue

          # New group
          if line.startswith(":"):
              current_group = line[1:].strip()
              groups[current_group] = []

          else:
              parts = line.split()

              # Safety check (some lines can be malformed)
              if len(parts) != 4:
                  continue

              a, b, c, d = parts
              groups[current_group].append((a, b, c, d))

  return groups

# Load the dataset
file_path = "./mikolov_analogy_dataset.txt"
analogy_data = load_analogies(file_path)
print(analogy_data.keys())

analogy_data_lower = {key: [(v.lower() for v in t) for t in value] for key, value in analogy_data.items()}

dict_keys(['capital-common-countries', 'capital-world', 'currency', 'city-in-state', 'family', 'gram1-adjective-to-adverb', 'gram2-opposite', 'gram3-comparative', 'gram4-superlative', 'gram5-present-participle', 'gram6-nationality-adjective', 'gram7-past-tense', 'gram8-plural', 'gram9-plural-verbs'])


In [ ]:
def predict_analogy(model, a, b, c):
    try:
        result = model.most_similar(
            positive=[b, c],
            negative=[a],
            topn=1
        )
        return result[0][0]
    except KeyError:
        return None

test = analogy_data['capital-common-countries'][2]
print("Test:", test[:-1])
print(predict_analogy(model_w2v, *test[:-1]))
print("Actual:", test[-1])
print()


Test: ('Athens', 'Greece', 'Beijing')
China
Actual: China

china


In [14]:
def evaluate_model(model, analogy_data, target_groups):
    results = {}
    total_count = sum([len(analogy_data[group]) for group in target_groups])
    abs_step = 0
    for group in target_groups:
        correct = 0
        total = 0
        skipped = 0

        count = len(analogy_data[group])
        for i, (a, b, c, d) in enumerate(analogy_data[group]):
            pred = predict_analogy(model, a, b, c)
            if pred is None:
                skipped += 1
                continue

            total += 1
            if pred == d:
                correct += 1
            print(f"{i+1}/{count} finished (Total Progress {abs_step / total_count * 100:.2f}%)")
            abs_step += 1

        accuracy = correct / total if total > 0 else 0

        results[group] = {
            "correct": correct,
            "total": total,
            "skipped": skipped,
            "accuracy": accuracy,
            "precision": accuracy,
            "recall": accuracy
        }
        print("Finished processing group", group)
    return results

def aggregate_results(results):
    correct = sum(v["correct"] for v in results.values())
    total = sum(v["total"] for v in results.values())
    skipped = sum(v["skipped"] for v in results.values())

    accuracy = correct / total if total > 0 else 0

    return {
        "correct": correct,
        "total": total,
        "skipped": skipped,
        "accuracy": accuracy,
        "precision": accuracy,
        "recall": accuracy
    }

def print_results(results, model_name):
    print(f"\n===== {model_name} =====\n")

    for group, stats in results.items():
        print(f"{group}")
        print(f"  Accuracy : {stats['accuracy']:.3f}")
        print(f"  Precision: {stats['precision']:.3f}")
        print(f"  Recall   : {stats['recall']:.3f}")
        print(f"  Correct  : {stats['correct']}")
        print(f"  Total    : {stats['total']}")
        print(f"  Skipped  : {stats['skipped']}\n")

In [15]:
target_groups = [
    "capital-common-countries",
    "capital-world",
    "currency",
    "city-in-state",
    "family",
    "gram1-adjective-to-adverb",
    "gram2-opposite",
    "gram3-comparative",
    "gram6-nationality-adjective",
    "gram8-plural"
]

In [ ]:

results_w2v = evaluate_model(model_w2v, analogy_data, target_groups)


1/506 finished (Total Progress 0.00%)
2/506 finished (Total Progress 0.01%)
3/506 finished (Total Progress 0.01%)
4/506 finished (Total Progress 0.02%)
5/506 finished (Total Progress 0.03%)
6/506 finished (Total Progress 0.03%)
7/506 finished (Total Progress 0.04%)
8/506 finished (Total Progress 0.05%)
9/506 finished (Total Progress 0.05%)
10/506 finished (Total Progress 0.06%)
11/506 finished (Total Progress 0.07%)
12/506 finished (Total Progress 0.07%)
13/506 finished (Total Progress 0.08%)
14/506 finished (Total Progress 0.09%)
15/506 finished (Total Progress 0.09%)
16/506 finished (Total Progress 0.10%)
17/506 finished (Total Progress 0.11%)
18/506 finished (Total Progress 0.11%)
19/506 finished (Total Progress 0.12%)
20/506 finished (Total Progress 0.13%)
21/506 finished (Total Progress 0.13%)
22/506 finished (Total Progress 0.14%)
23/506 finished (Total Progress 0.15%)
24/506 finished (Total Progress 0.15%)
25/506 finished (Total Progress 0.16%)
26/506 finished (Total Progress 0.

In [17]:
print_results(results_w2v, "Word2Vec")


===== Word2Vec =====

capital-common-countries
  Accuracy : 0.832
  Precision: 0.832
  Recall   : 0.832
  Correct  : 421
  Total    : 506
  Skipped  : 0

capital-world
  Accuracy : 0.791
  Precision: 0.791
  Recall   : 0.791
  Correct  : 3580
  Total    : 4524
  Skipped  : 0

currency
  Accuracy : 0.351
  Precision: 0.351
  Recall   : 0.351
  Correct  : 304
  Total    : 866
  Skipped  : 0

city-in-state
  Accuracy : 0.709
  Precision: 0.709
  Recall   : 0.709
  Correct  : 1749
  Total    : 2467
  Skipped  : 0

family
  Accuracy : 0.846
  Precision: 0.846
  Recall   : 0.846
  Correct  : 428
  Total    : 506
  Skipped  : 0

gram1-adjective-to-adverb
  Accuracy : 0.285
  Precision: 0.285
  Recall   : 0.285
  Correct  : 283
  Total    : 992
  Skipped  : 0

gram2-opposite
  Accuracy : 0.427
  Precision: 0.427
  Recall   : 0.427
  Correct  : 347
  Total    : 812
  Skipped  : 0

gram3-comparative
  Accuracy : 0.908
  Precision: 0.908
  Recall   : 0.908
  Correct  : 1210
  Total    : 1332
  S

In [18]:
w2v_overall = aggregate_results(results_w2v)
print("Word2Vec overall:", w2v_overall)


Word2Vec overall: {'correct': 10957, 'total': 14936, 'skipped': 0, 'accuracy': 0.7335966791644349, 'precision': 0.7335966791644349, 'recall': 0.7335966791644349}


In [26]:
results_glove = evaluate_model(model_glove, analogy_data_lower, target_groups)


1/506 finished (Total Progress 0.00%)
2/506 finished (Total Progress 0.01%)
3/506 finished (Total Progress 0.01%)
4/506 finished (Total Progress 0.02%)
5/506 finished (Total Progress 0.03%)
6/506 finished (Total Progress 0.03%)
7/506 finished (Total Progress 0.04%)
8/506 finished (Total Progress 0.05%)
9/506 finished (Total Progress 0.05%)
10/506 finished (Total Progress 0.06%)
11/506 finished (Total Progress 0.07%)
12/506 finished (Total Progress 0.07%)
13/506 finished (Total Progress 0.08%)
14/506 finished (Total Progress 0.09%)
15/506 finished (Total Progress 0.09%)
16/506 finished (Total Progress 0.10%)
17/506 finished (Total Progress 0.11%)
18/506 finished (Total Progress 0.11%)
19/506 finished (Total Progress 0.12%)
20/506 finished (Total Progress 0.13%)
21/506 finished (Total Progress 0.13%)
22/506 finished (Total Progress 0.14%)
23/506 finished (Total Progress 0.15%)
24/506 finished (Total Progress 0.15%)
25/506 finished (Total Progress 0.16%)
26/506 finished (Total Progress 0.

In [27]:
print_results(results_glove, "GloVe")


===== GloVe =====

capital-common-countries
  Accuracy : 0.949
  Precision: 0.949
  Recall   : 0.949
  Correct  : 480
  Total    : 506
  Skipped  : 0

capital-world
  Accuracy : 0.960
  Precision: 0.960
  Recall   : 0.960
  Correct  : 4342
  Total    : 4524
  Skipped  : 0

currency
  Accuracy : 0.158
  Precision: 0.158
  Recall   : 0.158
  Correct  : 137
  Total    : 866
  Skipped  : 0

city-in-state
  Accuracy : 0.593
  Precision: 0.593
  Recall   : 0.593
  Correct  : 1463
  Total    : 2467
  Skipped  : 0

family
  Accuracy : 0.881
  Precision: 0.881
  Recall   : 0.881
  Correct  : 446
  Total    : 506
  Skipped  : 0

gram1-adjective-to-adverb
  Accuracy : 0.226
  Precision: 0.226
  Recall   : 0.226
  Correct  : 224
  Total    : 992
  Skipped  : 0

gram2-opposite
  Accuracy : 0.273
  Precision: 0.273
  Recall   : 0.273
  Correct  : 222
  Total    : 812
  Skipped  : 0

gram3-comparative
  Accuracy : 0.881
  Precision: 0.881
  Recall   : 0.881
  Correct  : 1174
  Total    : 1332
  Skip

In [28]:
glove_overall = aggregate_results(results_glove)
print("GloVe overall:", glove_overall)

GloVe overall: {'correct': 11008, 'total': 14936, 'skipped': 0, 'accuracy': 0.7370112479914301, 'precision': 0.7370112479914301, 'recall': 0.7370112479914301}


In [ ]:
# Show similarity for opposite words (antonyms)
print("increase->decrease (antonym):", model_w2v.similarity("increase", "decrease"))  # Embeddings are super close!

print("dude->guy (synonym):", model_w2v.similarity("dude", "guy"))

# The antonyms are actually closer!

increase->decrease (antonym): 0.8370319
dude->guy (synonym): 0.7975314


In [45]:
# New Analogy 1: Reversing roles
role_dataset = [
    ("buy", "sell", "borrow", "lend"),
    ("teacher", "student", "employer", "employee"),
    ("doctor", "patient", "lawyer", "client")
]
print("=== Role Analogy ===")
for name, model in zip(("Word2Vec", "GloVe"), (model_w2v, model_glove)):
    print("--", name, "--")
    for question in role_dataset:
        print("Question:", question, "- Result:", predict_analogy(model, *question[:-1]), "- Correct:", question[-1])
print("")

# New Analogy 2: Different levels
level_dataset = [
    ("warm", "hot", "cool", "cold"),
    ("good", "better", "bad", "worse"),
    ("big", "huge", "small", "tiny")
]
print("=== Levels Analogy ===")
for name, model in zip(("Word2Vec", "GloVe"), (model_w2v, model_glove)):
    print("--", name, "--")
    for question in level_dataset:
        print("Question:", question, "- Result:", predict_analogy(model, *question[:-1]), "- Correct:", question[-1])

=== Role Analogy ===
-- Word2Vec --
Question: ('buy', 'sell', 'borrow', 'lend') - Result: borrowing - Correct: lend
Question: ('teacher', 'student', 'employer', 'employee') - Result: employers - Correct: employee
Question: ('doctor', 'patient', 'lawyer', 'client') - Result: Andrew_Levander - Correct: client
-- GloVe --
Question: ('buy', 'sell', 'borrow', 'lend') - Result: borrowing - Correct: lend
Question: ('teacher', 'student', 'employer', 'employee') - Result: employers - Correct: employee
Question: ('doctor', 'patient', 'lawyer', 'client') - Result: lawyers - Correct: client

=== Levels Analogy ===
-- Word2Vec --
Question: ('warm', 'hot', 'cool', 'cold') - Result: Hot - Correct: cold
Question: ('good', 'better', 'bad', 'worse') - Result: worse - Correct: worse
Question: ('big', 'huge', 'small', 'tiny') - Result: large - Correct: tiny
-- GloVe --
Question: ('warm', 'hot', 'cool', 'cold') - Result: hottest - Correct: cold
Question: ('good', 'better', 'bad', 'worse') - Result: worse -